# causal-probe: Round 2 / D2: Gemma-2-2B follow-up (corrected steering controls and adjacency test)

Read `KAGGLE.md` ("Round 2") in the repository first. **What this run does:** ONLY needed if notebook D was launched from a commit before cdb08d7: steering controls shuffled within hypothesis string, and the adjacency test with all its arms, at four sites. **Estimated time:** 1.5 to 2.5 hours on a T4.

**On Kaggle:** settings (right-hand panel) **Accelerator = GPU T4 x2**, **Internet = On**, the secret **`HF_TOKEN`** attached to THIS notebook (Gemma is gated; secrets are attached per notebook, and the first code cell stops at once if it is missing or has no access to the weights). Then *Save Version → Save & Run All (Commit)*. Output: the zip files in **`/kaggle/working/kaggle_out`** (Output tab of the saved version, folder `kaggle_out`).

**On Colab:** Runtime → Change runtime type → **T4 GPU**; then Runtime → **Run all**; the secret **`HF_TOKEN`** attached to THIS notebook (Gemma is gated; secrets are attached per notebook, and the first code cell stops at once if it is missing or has no access to the weights). Keep the browser tab open until the last cell has finished: a free Colab session stops when the tab is closed or idle for long, and its files are deleted. A cell near the end downloads the zip files from `/content/out` through the browser (allow multiple downloads when asked).

The notebook works on either platform: the first code cell detects where it is running, writes the archives to the place that
platform keeps, and stops before any GPU time is spent if it cannot. **The last cell lists the output directory with sizes,
writes `STATUS.txt` there (first line `OK`, `FAILED` or `SKIPPED`) and prints a banner. Download `STATUS.txt` with the zip files.**

Everything heavy happens in `scripts/gpu_round2.py`: each stage is time-limited, skipped if already done, and allowed to fail without
stopping the rest. Afterwards put the downloaded files into `kaggle_in/r2b_gemma/` on the laptop (see `notes/kaggle_ingest.md`).

In [ ]:
# ---- settings ---------------------------------------------------------------
COMMIT      = "df7313e0069b4d0b47e59a68511611150ca07b25"   # the code this notebook runs (pinned)
RUN_NAME    = "r2b_gemma"
MODELS      = "gemma2b"
JOBS        = "cache,controls,controls_hyp,adjacency,export"
FALLBACK    = ""
EXTRA_ARGS  = ""   # further options of scripts/gpu_round2.py (size of the intervention jobs)
NEEDS_TOKEN = True
GATED_REPOS = "google/gemma-2-2b-it"   # the first cell checks that the token has access to these before anything is installed
MIN_GPUS    = 1
SMOKE_TEST  = False        # True: a few items per stage (about 10 minutes) to check the plumbing
SKIP_STAGES = ""           # comma-separated stage-name prefixes to skip, e.g. "r2.qwen3b.patch"

In [ ]:
import os, subprocess, sys, json, shutil, time
t_start = time.time()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else ""
print(gpu or "NO GPU FOUND: switch the accelerator to a GPU (Kaggle: notebook settings; Colab: Runtime -> Change runtime type). The run would be far too slow on CPU.")

# ---- which platform is this, and where must the output go? --------------------
# Kaggle keeps ONLY /kaggle/working after "Save & Run All". Colab keeps nothing after the session, so the
# archives are downloaded through the browser from /content/out. Kaggle's image also has a /content folder,
# which is why Kaggle is tested first.
def choose_dirs(env, isdir):
    """-> platform, output directory, scratch directory, the path this platform keeps (None: no rule)"""
    if env.get("KAGGLE_KERNEL_RUN_TYPE") or isdir("/kaggle"):
        return "kaggle", "/kaggle/working/kaggle_out", "/kaggle/temp", "/kaggle/working"
    if env.get("COLAB_RELEASE_TAG") or env.get("COLAB_GPU") or isdir("/content"):
        return "colab", "/content/out", "/content/tmp", "/content/out"
    return "local", os.path.abspath("kaggle_out"), os.path.abspath("kaggle_tmp"), None

def under(path, parent):
    return (os.path.abspath(path) + "/").startswith(os.path.abspath(parent) + "/")

PLATFORM, WORK, TEMP, PERSIST = choose_dirs(os.environ, os.path.isdir)
os.makedirs(TEMP, exist_ok=True); os.makedirs(WORK, exist_ok=True)
# Stop here, before any GPU time is spent, if the archives would be written where the platform does not keep them.
if PERSIST and not under(WORK, PERSIST):
    raise RuntimeError(f"output directory {WORK} is not under {PERSIST}: on {PLATFORM} the archives would be lost")
with open(WORK + "/.write_test", "w") as f:
    f.write("ok")
os.remove(WORK + "/.write_test")
print("platform:", PLATFORM, "| archives will be written to", WORK,
      {"kaggle": "(kept by Kaggle: Output tab of the saved version, folder kaggle_out)",
       "colab": "(NOT kept after the session: the last cell hands them to the browser)"}.get(PLATFORM, ""))

# ---- Hugging Face token, from this platform's Secrets ---------------------------
# It is never printed or written to disk by this notebook. A secret is attached PER NOTEBOOK: on Kaggle,
# Add-ons -> Secrets -> tick HF_TOKEN for this notebook (a new or imported notebook starts with none attached).
def read_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    if PLATFORM == "kaggle":
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle Secrets"
    if PLATFORM == "colab":
        from google.colab import userdata
        return userdata.get("HF_TOKEN"), "Colab Secrets"
    return None, ""
try:
    _tok, _where = read_token()
    if not _tok:
        raise LookupError("empty")
    os.environ["HF_TOKEN"] = _tok
    del _tok
    print("HF_TOKEN loaded from", _where)
except Exception as e:
    print("No HF_TOKEN secret is attached to this notebook (%s)." % type(e).__name__)
    if NEEDS_TOKEN:
        raise RuntimeError("This run needs the HF_TOKEN secret (the Gemma weights are gated). Attach it to THIS notebook "
                           "(Kaggle: Add-ons -> Secrets -> tick HF_TOKEN; Colab: the key icon in the left bar) and run again.")
    print("This run does not need one: the Qwen weights are public and are downloaded anonymously.")

# ---- gated weights: does this token have access? Asked now, before anything is installed ----
def check_access(repo, token):
    """(True, "") access; (False, reason) refused by Hugging Face; (None, reason) the question could not be asked
    (no network, certificate store missing). One small request for the repository's config.json; redirects are not
    followed, so the token is sent to huggingface.co only."""
    import ssl, urllib.error, urllib.request
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *args, **kw):
            return None
    try:
        import certifi
        ctx = ssl.create_default_context(cafile=certifi.where())
    except Exception:
        ctx = ssl.create_default_context()
    req = urllib.request.Request("https://huggingface.co/" + repo + "/resolve/main/config.json", method="HEAD",
                                 headers={"Authorization": "Bearer " + token} if token else {})
    try:
        urllib.request.build_opener(NoRedirect, urllib.request.HTTPSHandler(context=ctx)).open(req, timeout=30)
        return True, ""
    except urllib.error.HTTPError as e:
        if 300 <= e.code < 400:
            return True, ""                       # a redirect to the file: access granted
        if e.code in (401, 403, 404):
            return False, {401: "the token was rejected, or the licence of this model has not been accepted with the token's account",
                           403: "this account has not been granted access to the model",
                           404: "repository or file not found"}[e.code]
        return None, "HTTP %d" % e.code
    except Exception as e:
        return None, "%s: %s" % (type(e).__name__, str(getattr(e, "reason", e))[:120])
for _repo in [r for r in globals().get("GATED_REPOS", "").split(",") if r]:
    _ok, _why = check_access(_repo, os.environ.get("HF_TOKEN", ""))
    if _ok is False:
        raise RuntimeError("No access to " + _repo + ": " + _why + ". Open https://huggingface.co/" + _repo + " while logged in to "
                           "the account the token belongs to, accept the licence, then run this notebook again. Nothing has been run or installed.")
    if _ok is None:      # not a refusal: do not stop a run over a check that could not be made; the download will show it
        print("WARNING: could not ask Hugging Face about access to", _repo, "(" + _why + "). Is Internet switched on? Continuing; "
              "if access is missing, the model download fails within the first minutes of the run.")
        continue
    print("access to", _repo, "confirmed")

# ---- GPUs: a model that is split over two cards needs both. Checked now, before anything is installed ----
_n_gpu = len([l for l in gpu.strip().splitlines()[1:] if l.strip()])
if PLATFORM != "local" and globals().get("MIN_GPUS", 1) > 1 and _n_gpu < MIN_GPUS:       # one-GPU runs keep the warning printed above
    raise RuntimeError("This run needs %d GPUs and this session has %d. On Kaggle set Accelerator = GPU T4 x2 (not P100, not a "
                       "single T4) and run again. Nothing has been run or installed." % (MIN_GPUS, _n_gpu))

os.environ["HF_HOME"] = TEMP + "/hf"                 # model downloads (not part of the output)
os.environ["CAUSAL_PROBE_CACHE"] = TEMP + "/cache"   # full activation cache (not part of the output)
os.environ["CAUSAL_PROBE_JOBS"] = "4" if (os.cpu_count() or 2) >= 4 else "2"
os.environ["PYTHONPATH"] = TEMP + "/causal-probe/src"   # the package is used straight from the checkout

In [ ]:
# ---- code at the pinned commit ------------------------------------------------
REPO_DIR = TEMP + "/causal-probe"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/skape007-ult/causal-probe.git", REPO_DIR], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR)
subprocess.run(["git", "checkout", "--quiet", COMMIT], cwd=REPO_DIR, check=True)
print("code at", subprocess.run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR, capture_output=True, text=True).stdout)

In [ ]:
# ---- pinned environment -------------------------------------------------------
# Level 1: a fresh Python 3.12 virtual environment with the full freeze from the laptop (requirements.txt).
# Level 2: the same venv with the pinned direct dependencies only (requirements-kaggle.txt).
# Level 3: this machine's own Python and torch, plus the missing packages (versions then differ; recorded in the results).
def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print("FAILED:", cmd, "\n", (r.stdout + r.stderr)[-1500:])
    return r.returncode == 0

VENV = TEMP + "/venv"
PY, level = None, None
extra = ""
if os.path.exists(VENV + "/bin/python") and os.path.exists(TEMP + "/install_level"):
    PY, level = VENV + "/bin/python", open(TEMP + "/install_level").read()
    if level.startswith("3"): PY = sys.executable
else:
    if sh(f"{sys.executable} -m pip install -q uv") and sh(f"uv venv --python 3.12 {VENV}"):
        if sh(f"uv pip install -q --python {VENV}/bin/python -r {REPO_DIR}/requirements.txt{extra} && uv pip install -q --python {VENV}/bin/python -e {REPO_DIR}"):
            PY, level = VENV + "/bin/python", "1 (full freeze)"
        elif sh(f"uv pip install -q --python {VENV}/bin/python -r {REPO_DIR}/requirements-kaggle.txt{extra} && uv pip install -q --python {VENV}/bin/python -e {REPO_DIR}"):
            PY, level = VENV + "/bin/python", "2 (pinned direct dependencies)"
    if PY is None:
        sh(f"{sys.executable} -m pip install -q sae-lens peft accelerate{extra}")
        PY, level = sys.executable, "3 (this machine's own packages; versions differ from the laptop)"
    open(TEMP + "/install_level", "w").write(level)
print("install level:", level, "| python:", PY, "| %.0f min so far" % ((time.time() - t_start) / 60))
r = subprocess.run([PY, "-c", "import torch, transformers, sklearn, causal_probe; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())"], capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])
if "4bit" in MODELS:
    # 4-bit loading needs bitsandbytes. Installed separately and allowed to fail, so that it cannot
    # change the pinned packages above; without it the runner falls back to float16 on two GPUs.
    if PY != sys.executable:
        sh(f"uv pip install -q --no-deps --python {PY} bitsandbytes")
    else:
        sh(f"{sys.executable} -m pip install -q --no-deps bitsandbytes")
    r = subprocess.run([PY, "-c", "import torch, bitsandbytes; print('bitsandbytes', bitsandbytes.__version__, '| torch still', torch.__version__)"], capture_output=True, text=True)
    print(r.stdout or "bitsandbytes not importable (the float16 two-GPU fallback will be used): " + r.stderr[-300:])

In [ ]:
# ---- run (resumable: re-running this cell skips finished stages) ---------------
cmd = [PY, "-u", "scripts/gpu_round2.py", "--run", RUN_NAME, "--models", MODELS, "--jobs", JOBS, "--out", WORK]
if FALLBACK: cmd += ["--fallback", FALLBACK]
cmd += EXTRA_ARGS.split()
if SMOKE_TEST: cmd.append("--tiny")
if SKIP_STAGES: cmd += ["--skip", SKIP_STAGES]
print(" ".join(cmd))
p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print("runner exit code", p.returncode, "| total %.0f min" % ((time.time() - t_start) / 60))

In [ ]:
# ---- what ran, and which archives to expect -----------------------------------
# Nothing in this cell may stop the notebook: the last cell must run whatever happened before.
RUN = RUN_NAME + ("_tiny" if SMOKE_TEST else "")      # a smoke test runs under its own name and results folder
s, expected, zips, not_finished, skipped = {}, ["results_" + RUN + ".zip"], [], "", ""
try:
    st = REPO_DIR + "/results/" + ("_tiny/" if SMOKE_TEST else "") + "round2_status_" + RUN + ".json"
    s = json.load(open(st)) if os.path.exists(st) else {}
    for k in s.get("planned", []):
        v = s["stages"].get(k)
        print("not run         " + k if v is None else ("ok     " if v["ok"] else "FAILED ") + f"{v['minutes']:6.1f} min  {k}" + ("" if v["ok"] else "   <- " + v["detail"]))
    print("export:", json.dumps(s.get("export", {}).get("zip_megabytes", {})))
    if not s.get("finished"):
        not_finished = "the runner did not reach its export step (no finished flag in " + st + ")"
    # expected: the small results archive, every archive the runner recorded, and one activation archive per model whose cache stage succeeded
    expected += list(s.get("export", {}).get("zip_megabytes", {}))
    for m in s.get("models_run", []):
        if any(s.get("stages", {}).get(f"r2.{m}.{j}", {}).get("ok") for j in ("cache", "behaviour.cache")):
            expected.append(f"cache_{RUN}_{m}.zip")
    expected = list(dict.fromkeys(expected))
    zips = sorted(f for f in os.listdir(WORK) if f.endswith(".zip"))
    # a run that decided not to run (no access, not enough GPU memory): see the runner's preflight
    skipped = "; ".join(f"{m}: {why}" for m, why in s.get("skipped", {}).items()) if not s.get("models_run") else ""
except Exception as e:
    not_finished = "the runner's status could not be read (%s: %s)" % (type(e).__name__, e)
# the runner itself ended with an error (an unknown option, a refused output directory): whatever status is on disk is from an earlier attempt
_rc = getattr(globals().get("p"), "returncode", None)
if _rc not in (0, None):
    not_finished = "the runner ended with exit code %s (see the output of the cell above)" % _rc
elif _rc is None:
    not_finished = not_finished or "the run cell was not executed in this session"

In [ ]:
# ---- final check: are the archives where this platform keeps them? --------------
# Writes STATUS.txt into the output directory (first line: OK, FAILED or SKIPPED), prints a banner, and ends normally:
# it never raises, because a version whose last cell failed might not keep its output. The ingest on the laptop
# refuses a run whose STATUS.txt is missing or does not say OK.
import datetime, zipfile

def check_outputs(work, persist, expected):
    """List the output directory with sizes. Returns the list of problems (empty: all expected archives are in place)."""
    problems, total = [], 0
    print("output directory:", work)
    for root, _, fs in sorted(os.walk(work)):
        for f in sorted(fs):
            p = os.path.join(root, f); n = os.path.getsize(p); total += n
            print(f"  {n / 1e6:10.1f} MB  {os.path.relpath(p, work)}")
    print(f"  {total / 1e6:10.1f} MB  in total")
    if persist and not under(work, persist):
        problems.append(f"the output directory {work} is not under {persist}, the only place this platform keeps")
    if not expected:
        problems.append("no archive was expected: the runner left no record of what it exported")
    for f in expected:
        p = os.path.join(work, f)
        if not os.path.isfile(p):
            problems.append("expected archive is missing: " + p)
        elif not zipfile.is_zipfile(p) or os.path.getsize(p) < 1000:
            problems.append("expected archive is empty or not a zip file: " + p)
    return problems

def write_status(work, state, run, commit, platform, expected, problems):
    """STATUS.txt in the output directory, and a copy inside the small results archive (results/round2_output_status_<run>.txt),
    so that the status travels with the download even if STATUS.txt itself is not downloaded."""
    lines = [state, "run: " + run, "commit: " + commit, "platform: " + platform, "output directory: " + work,
             "written: " + datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "archives:"]
    for f in expected:
        p = os.path.join(work, f)
        lines.append("  " + f + "  " + (str(os.path.getsize(p)) + " bytes" if os.path.isfile(p) else "MISSING"))
    lines.append("problems:" + ("" if problems else " none"))
    lines += ["  - " + q for q in problems]
    text = "\n".join(lines) + "\n"
    with open(os.path.join(work, "STATUS.txt"), "w") as f:
        f.write(text)
    member = "results/round2_output_status_" + run + ".txt"
    for f in expected:
        p = os.path.join(work, f)
        if f.startswith("results") and os.path.isfile(p) and zipfile.is_zipfile(p):
            tmp = p + ".tmp"                      # rewritten rather than appended to, so that re-running this cell replaces the copy
            with zipfile.ZipFile(p) as zi, zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zo:
                for it in zi.infolist():
                    if it.filename != member:
                        zo.writestr(it, zi.read(it.filename))
                zo.writestr(member, text)
            os.replace(tmp, p)
    return text

try:
    problems = ([not_finished] if not_finished and not skipped else []) + check_outputs(WORK, PERSIST, expected)
    OUTPUT_STATE = "SKIPPED" if skipped else ("FAILED" if problems else "OK")
    if skipped:
        problems = ["the run skipped itself: " + skipped] + problems
    status_text = write_status(WORK, OUTPUT_STATE, RUN, COMMIT, PLATFORM, expected, problems)
except Exception as e:                            # the check itself must not be what loses the output
    OUTPUT_STATE, problems, status_text = "FAILED", ["the output check itself failed: %s: %s" % (type(e).__name__, e)], ""
    try:
        with open(os.path.join(WORK, "STATUS.txt"), "w") as f:
            f.write("FAILED\nrun: %s\nproblems:\n  - %s\n" % (RUN, problems[0]))
    except Exception:
        pass
bar = ("#" if OUTPUT_STATE == "OK" else "!") * 100
print("\n" + bar)
if OUTPUT_STATE == "OK":
    print("OUTPUT CHECK PASSED:", len(expected), "archive(s) and STATUS.txt in", WORK)
    print({"kaggle": "Download the zip files and STATUS.txt from the Output tab of this saved version (folder kaggle_out).",
           "colab": "The archives and STATUS.txt are handed to the browser below; if that is blocked, download them from the Files panel now."}.get(PLATFORM, ""))
elif OUTPUT_STATE == "SKIPPED":
    print("RUN SKIPPED, nothing to download. Reason: " + skipped)
else:
    print("OUTPUT CHECK FAILED. Save what is listed above by hand before this session ends. The ingest will refuse this run.")
for q in problems:
    print("  - " + q)
print(bar)
print("STATUS.txt, first line:", OUTPUT_STATE)

# ---- Colab only: hand the archives and STATUS.txt to the browser ------------------
# Here, after the status has been written (its copy is inside the results archive). Colab deletes /content when the
# session ends. If the browser blocks the downloads, open the Files panel (folder icon, left) and download from /content/out.
if PLATFORM == "colab":
    try:
        from google.colab import files
        for f in [f for f in expected if os.path.isfile(os.path.join(WORK, f))] + ["STATUS.txt"]:
            files.download(os.path.join(WORK, f))
    except Exception as e:
        print("automatic download not available here (%s): use the Files panel" % type(e).__name__)